# CLI

Command-line interface backed by [python-fire](https://github.com/google/python-fire). Invoke via `python -m sugar <command> ...` or the `sugar` console script.

In [ ]:
#| default_exp cli

In [ ]:
#| export

import fire
from sugar.chains import get_chain
from sugar.deposit import DepositQuote
from sugar.helpers import ADDRESS_ZERO, normalize_address

In [ ]:
#| export

# Fire's literal_eval parses `0x...` as a hex int — unmangle before checksum/normalize.
def _addr(v): return v if v is None else normalize_address('0x' + format(v, '040x') if isinstance(v, int) else v)

def _new_pool_spec(c, token0, token1, pool_type, tick_spacing):
    t0, t1 = c.get_token(token0), c.get_token(token1)
    if t0 is None or t1 is None: raise SystemExit('token0 or token1 not known to Sugar')
    pool_type = str(pool_type).lower()
    if pool_type == 'cl':
        if tick_spacing is None: raise SystemExit('--pool-type cl requires --tick-spacing N')
        return c.pool_spec(t0, t1, tick_spacing=int(tick_spacing))
    if pool_type in ('stable', 'volatile'):
        if tick_spacing is not None: raise SystemExit('--tick-spacing is CL-only')
        return c.pool_spec(t0, t1, stable=(pool_type == 'stable'))
    raise SystemExit(f'invalid --pool-type: {pool_type} (expected cl/stable/volatile)')

def _resolve_pool(c, *, pool=None, token0=None, token1=None, pool_type=None, tick_spacing=None):
    """--pool ADDR (existing) XOR (--token0 --token1 --pool-type [--tick-spacing]) (new)."""
    pool, token0, token1 = _addr(pool), _addr(token0), _addr(token1)
    if pool is not None:
        if any(x is not None for x in (token0, token1, pool_type, tick_spacing)):
            raise SystemExit('--pool cannot be combined with --token0/--token1/--pool-type/--tick-spacing')
        p = c.get_pool_by_address(pool)
        if p is None: raise SystemExit(f'pool {pool} not found')
        return p
    if not token0 or not token1 or not pool_type:
        raise SystemExit('new pool requires --token0, --token1, and --pool-type {cl,stable,volatile}')
    return _new_pool_spec(c, token0, token1, pool_type, tick_spacing)

def _one_side(a0, a1, ctx):
    """Return the `amount_tokenN` kwarg for whichever side is set, raising if both/neither."""
    if (a0 is None) == (a1 is None): raise SystemExit(f'{ctx} requires exactly one of --amount0 / --amount1')
    return {'amount_token0': a0} if a0 is not None else {'amount_token1': a1}

def _build_quote(c, p, *, amount0, amount1, price_lower, price_upper, tick_lower, tick_upper, initial_price):
    """Resolve a DepositQuote for either basic or CL, new or existing pool."""
    is_new = p.lp == ADDRESS_ZERO
    a0 = p.token0.parse_units(float(amount0)) if amount0 is not None else None
    a1 = p.token1.parse_units(float(amount1)) if amount1 is not None else None
    if p.is_cl:
        kwargs = _one_side(a0, a1, 'CL deposit')
        if is_new and initial_price is None: raise SystemExit('new CL pool requires --initial-price')
        if initial_price is not None: kwargs['initial_price'] = float(initial_price)
        if tick_lower is not None or tick_upper is not None:
            kwargs['tick_lower'], kwargs['tick_upper'] = tick_lower, tick_upper
        else:
            kwargs['price_lower'] = float(price_lower) if price_lower is not None else None
            kwargs['price_upper'] = float(price_upper) if price_upper is not None else None
        return c.quote_concentrated_deposit(p, **kwargs)
    if any(x is not None for x in (price_lower, price_upper, tick_lower, tick_upper, initial_price)):
        raise SystemExit('basic deposits do not accept CL flags')
    if is_new:
        if a0 is None or a1 is None: raise SystemExit('new basic pool requires both --amount0 and --amount1')
        return DepositQuote(pool=p, amount_token0=a0, amount_token1=a1)
    return c.quote_basic_deposit(p, **_one_side(a0, a1, 'existing basic pool deposit'))

In [ ]:
#| export

class CLI:
    """Sugar SDK command-line interface.

    Default is dry-run (no broadcast). Pass --broadcast to send the tx.
    Amounts are human-readable (e.g. `0.5` → `0.5 * 10^decimals`).
    `--chain` takes a numeric chain id (10, 8453, 130, 1135).

    Examples:
        # preview a basic deposit
        python -m sugar deposit --chain=10 --pool=0xd25711... --amount0=0.0001 --amount1=1

        # broadcast a CL deposit, range by prices
        python -m sugar deposit --chain=10 --pool=0x478946... --amount1=0.01 \
            --price-lower=2900 --price-upper=3200 --broadcast

        # create a new stable basic pool
        python -m sugar deposit --chain=8453 --token0=0xT0 --token1=0xT1 --pool-type=stable \
            --amount0=100 --amount1=100 --broadcast"""

    def deposit(self, *, chain: int, pool: str = None,
                token0: str = None, token1: str = None, pool_type: str = None, tick_spacing=None,
                amount0=None, amount1=None,
                price_lower=None, price_upper=None, tick_lower=None, tick_upper=None,
                initial_price=None, slippage: float = 0.01, deadline_minutes: float = 30,
                broadcast: bool = False):
        """Deposit liquidity. Dry-run returns unsigned tx(s); --broadcast returns the receipt."""
        with get_chain(str(chain), threading_max_workers=1) as c:
            p = _resolve_pool(c, pool=pool, token0=token0, token1=token1,
                              pool_type=pool_type, tick_spacing=tick_spacing)
            q = _build_quote(c, p, amount0=amount0, amount1=amount1,
                             price_lower=price_lower, price_upper=price_upper,
                             tick_lower=tick_lower, tick_upper=tick_upper,
                             initial_price=initial_price)
            r = c.deposit(q, delay_in_minutes=deadline_minutes, slippage=slippage, dry_run=not broadcast)
            if not broadcast: return r
            return {'tx': r.transactionHash.hex(), 'status': r.status, 'gas': r.gasUsed, 'block': r.blockNumber}

In [ ]:
#| export

def main():
    from dotenv import load_dotenv
    load_dotenv()
    fire.Fire(CLI)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()